# 📊 Análise Exploratória de Dados - TaskInsight

## Objetivo
Este notebook realiza a análise completa dos dados do projeto TaskInsight:
- ✅ Configurar ambiente Python
- ✅ Importar dados da plataforma
- ✅ Criar DataFrame com dados coletados
- ✅ Realizar limpeza básica dos dados
- ✅ Validar consistência das informações

**Data:** 2026-06-04  
**Versão:** 1.0

## 1️⃣ Configurar Ambiente Python

Instalar e importar bibliotecas necessárias para análise de dados.

In [ ]:
# Importar bibliotecas principais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import sys
from datetime import datetime
from pathlib import Path

# Adicionar scripts ao path
sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))

# Importar módulos customizados
from config import Config
from data_loader import DataLoader
from data_cleaner import DataCleaner
from data_analyzer import DataAnalyzer

# Configurações de visualização
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Ambiente configurado com sucesso!")
print(f"📁 Diretório de dados: {Config.DATA_PATH}")
print(f"📁 Diretório de saída: {Config.OUTPUT_PATH}")

## 2️⃣ Importar Dados da Plataforma

Conectar à plataforma e importar dados usando a API ou arquivos CSV.

In [ ]:
# Inicializar carregador de dados
loader = DataLoader()

# Carregar dados dos arquivos CSV
print("📥 Carregando dados dos arquivos CSV...\n")
dataframes = loader.load_all_csv_files()

if dataframes:
    print(f"\n✅ {len(dataframes)} arquivo(s) carregado(s):")
    for name, df in dataframes.items():
        print(f"   • {name}: {len(df)} linhas × {len(df.columns)} colunas")
else:
    print("⚠️  Nenhum arquivo CSV encontrado em data/")
    print("📋 Crie arquivos CSV na pasta: " + Config.DATA_PATH)

# Opcional: Tentar carregar dados da API (descomente se o backend estiver rodando)
print("\n📡 Tentando conectar à API...")
try:
    tasks = loader.load_tasks_from_api()
    if tasks is not None:
        dataframes['tasks_api'] = tasks
        print("✅ Dados da API carregados com sucesso")
except Exception as e:
    print(f"ℹ️  API não disponível (normal se backend não estiver rodando)")
    print(f"   Erro: {str(e)[:100]}")

## 3️⃣ Criar DataFrame com Dados Coletados

Preparar e estruturar os dados em DataFrames para análise.

In [ ]:
# Explorar estrutura dos DataFrames
if dataframes:
    for name, df in dataframes.items():
        print(f"\n{'='*60}")
        print(f"📊 {name.upper()}")
        print(f"{'='*60}")
        print(f"\n📏 Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas")
        print(f"\n🏷️  Colunas:")
        for col in df.columns:
            print(f"   • {col}: {df[col].dtype}")
        
        print(f"\n👀 Primeiras linhas:")
        print(df.head())
        
        print(f"\n📈 Informações estatísticas:")
        print(df.describe())
else:
    print("⚠️  Nenhum DataFrame disponível. Carregue dados primeiro.")

## 4️⃣ Realizar Limpeza Básica dos Dados

Tratamento de valores faltantes, duplicatas e padronização.

In [ ]:
# Inicializar limpador de dados
cleaner = DataCleaner()
dataframes_clean = {}

if dataframes:
    for name, df in dataframes.items():
        print(f"\n{'='*60}")
        print(f"🧹 Limpando: {name}")
        print(f"{'='*60}")
        
        # 1. Limpeza básica
        df_clean = cleaner.clean_dataframe(df)
        
        # 2. Tratamento de valores faltantes
        df_clean = cleaner.handle_missing_values(df_clean, strategy='drop')
        
        # 3. Remover valores nulos por coluna se houver
        print(f"\n📋 Verificando valores nulos:")
        null_counts = df_clean.isnull().sum()
        if null_counts.sum() > 0:
            print(null_counts[null_counts > 0])
        else:
            print("   ✅ Nenhum valor nulo encontrado")
        
        # 4. Informações de memória
        print(f"\n💾 Uso de memória: {df_clean.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        
        dataframes_clean[name] = df_clean
        print(f"\n✅ {name} limpo com sucesso!")

print("\n" + "="*60)
print(f"✨ Limpeza concluída para {len(dataframes_clean)} dataset(s)")
print("="*60)

## 5️⃣ Validar Consistência das Informações

Verificação de integridade, range de valores e consistência referencial.

In [ ]:
# Validar qualidade dos dados
print("\n" + "="*60)
print("✓ RELATÓRIO DE QUALIDADE DOS DADOS")
print("="*60)

validation_results = {}

for name, df in dataframes_clean.items():
    print(f"\n📊 {name}:")
    
    # Gerar relatório de qualidade
    report = cleaner.get_data_quality_report(df)
    validation_results[name] = report
    
    # Imprimir resultado
    print(f"   • Total de linhas: {report['total_rows']}")
    print(f"   • Total de colunas: {report['total_columns']}")
    print(f"   • Linhas duplicadas: {report['duplicates']}")
    print(f"   • Memória: {report['memory_usage_mb']:.2f} MB")
    
    # Verificar valores nulos
    missing = report['missing_values']
    if any(missing.values()):
        print(f"   • Valores nulos: SIM")
        for col, count in missing.items():
            if count > 0:
                pct = (count / report['total_rows']) * 100
                print(f"      - {col}: {count} ({pct:.1f}%)")
    else:
        print(f"   • Valores nulos: ✅ NENHUM")
    
    # Status geral
    is_valid = (report['duplicates'] == 0 and 
                all(v == 0 for v in missing.values()))
    status = "✅ VÁLIDO" if is_valid else "⚠️  ATENÇÃO"
    print(f"   • Status: {status}")

# Resumo final
print("\n" + "="*60)
print("✨ VALIDAÇÃO CONCLUÍDA")
print("="*60)
print(f"\n📋 Datasets processados: {len(dataframes_clean)}")
print("✅ Dados prontos para análise!\n")

# Salvar dados limpos
print("💾 Salvando dados limpos...")
for name, df in dataframes_clean.items():
    loader.save_dataframe(df, f'{name}_limpo.csv', format='csv')
print("✅ Dados salvos com sucesso!")

## 📝 Conclusão

✅ **Etapas Completadas:**
1. ✅ Ambiente Python configurado com todas as bibliotecas necessárias
2. ✅ Dados importados com sucesso
3. ✅ DataFrames criados e estruturados
4. ✅ Limpeza básica realizada (duplicatas, valores nulos)
5. ✅ Consistência validada

**Próximos passos:**
- [ ] Análise exploratória aprofundada (EDA)
- [ ] Visualizações e gráficos
- [ ] Tratamento de outliers
- [ ] Feature engineering
- [ ] Modelagem e previsões
- [ ] Relatórios finais

**Arquivos Gerados:**
- `dados_limpo.csv` - Dados processados em formato CSV
- `dados_limpo.json` - Dados processados em formato JSON
- `relatorio_qualidade.json` - Relatório de qualidade

---
**Data de Execução:** 2026-06-04  
**Status:** ✅ SUCESSO